# Prediction quality in crypt, neck and villus regions

**Role:** Analysis.

Load one all-marker model or all its saved validation folds and measure curvature prediction error, bias and variance by anatomical region. Region labels are analysis annotations, never model inputs. Neck interpretation requires circumference-profile support; missing or undetected crypt annotations remain explicit, especially at small N. This notebook contains no marker-coherence analysis and performs no training.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select saved models and anatomical annotations

Choose matching fold/seed variants from one trained family. The default annotation file uses previously qualified crypt circumference profiles. Other datasets can supply a node-keyed CSV with `organoid_str`, `node`, `region`; only qualified neck labels should be called neck.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'region_predictability'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# To analyze all folds, select the same family/depth/seed/subset in the catalog:
first = run.records.set_index('key').loc[MODEL_KEY]
MODEL_KEYS = run.records[(run.records.name==first['name']) & (run.records.depth==first.depth)
                        & (run.records.seed==first.seed) & (run.records.subset=='all')
                        & (run.records.signal=='intact')].key.tolist()  # Checkpoint keys included in the comparison.
for parameter in ['hidden_dim','rate']:
    if parameter in run.records:
        value=first[parameter]
        matching=run.records[parameter].isna() if pd.isna(value) else run.records[parameter].eq(value)
        MODEL_KEYS=[key for key in MODEL_KEYS if key in set(run.records.loc[matching,'key'])]
REGION_TABLE = ROOT/'results_experiments/ki67_observed_neighborhoods/exclusive_v2_profile_necks/nodes.csv.gz'  # Node-keyed anatomical annotation table; use qualified necks.
annotations = pd.read_csv(REGION_TABLE)
if annotations.duplicated(['organoid_str','node']).any():
    raise ValueError('Region annotation keys must be unique')
# The qualified table only assigns finite axis positions to profile-supported crypts.
if 'qualified_neck' in annotations:
    qualified = annotations.qualified_neck.astype(str).str.lower().eq('true')
    annotations.loc[(annotations.region=='neck_band') & ~qualified,'region']='unqualified_boundary'
annotations['region'] = annotations.region.replace({'crypt_body':'crypt','neck_band':'neck','beyond_neck':'villus'})
display(annotations.groupby('region').agg(cells=('node','size'),organoids=('organoid_str','nunique')))

## Predict held-out cells and join regions

Use the saved validation membership, target transforms and baseline offsets. Models must receive the complete marker panel. Never assign unmatched cells to villus; record annotation-unavailable instead.

In [ ]:
frames=[]
for key in MODEL_KEYS:
    chosen=run.select(key,device=DEVICE)
    if chosen['marker_names'] != chosen.get('all_marker_names', chosen['marker_names']):
        raise ValueError('Region predictability expects an all-marker model')
    frame=prediction_table(chosen,device=DEVICE).assign(key=key)
    frame=frame.merge(annotations[['organoid_str','node','region']],on=['organoid_str','node'],how='left',validate='many_to_one')
    frame['region']=frame.region.fillna('annotation_unavailable')
    frames.append(frame)
cells=pd.concat(frames,ignore_index=True)
# Repeated organoids across seeds/models are kept separate, never pooled as extra biology.
rows=[]
for (key,org,region),frame in cells.groupby(['key','organoid_str','region']):
    error_z=frame.pred_transformed-frame.y_transformed
    rows.append(dict(key=key,organoid_str=org,region=region,n_cells=len(frame),N=frame.n_cells.iloc[0],
        mse=frame.squared_error.mean(),bias=frame.error.mean(),
        measured_variance=frame.y_true.var(ddof=0),predicted_mean_variance=frame.y_pred.var(ddof=0),
        residual_variance=frame.error.var(ddof=0),mean_model_variance_z=frame.variance_transformed.mean(),
        standardized_squared_error=np.mean(error_z**2/frame.variance_transformed.clip(lower=1e-12))))
regional=pd.DataFrame(rows)
display(regional.groupby('region').agg(organoids=('organoid_str','nunique'),cells=('n_cells','sum'),mse=('mse','mean'),bias=('bias','mean')))
regional.to_csv(OUTPUT_DIR/'regional_metrics.csv',index=False)
cells.to_csv(OUTPUT_DIR/'regional_predictions.csv.gz',index=False)

## Regional error and variation

Physical MSE and empirical variance are averaged equally across organoids with cells in that region. Model variance and standardized error are separately reported in transformed target units; the nonlinear inverse does not preserve a Gaussian distribution. Neck/detection support changes with N, so regional curves are conditional on detectable, qualified anatomy.

In [ ]:
metrics=['mse','measured_variance','residual_variance','mean_model_variance_z']
fig,axes=plt.subplots(1,4,figsize=(17,4))
for ax,metric in zip(axes,metrics):
    regional.groupby('region')[metric].mean().plot.bar(ax=ax)
    ax.set(ylabel=metric,xlabel='Region');ax.tick_params(axis='x',rotation=70)
plt.tight_layout();plt.show()
regional['size_bin']=pd.qcut(regional.N,4,duplicates='drop')
display(regional.groupby(['region','size_bin'],observed=True).agg(organoids=('organoid_str','nunique'),mse=('mse','mean'),variance=('measured_variance','mean')))
display(regional.groupby('region')[['bias','standardized_squared_error']].mean())